# 4. Transformer Architecture: BERT (Bidirectional Encoder from Transformers)
**CSE 4122 — Natural Language Processing Laboratory**  
*Department of Computer Science and Engineering, Khulna University of Engineering & Technology (KUET)*

---

### Overview
This notebook fine-tunes **BERT** (`bert-base-uncased`) for sarcastic text classification:
- **Pretrained Representations**: 12-layer transformer encoder pretrained on BookCorpus and English Wikipedia.
- **WordPiece Tokenization**: Preserves subword units and handles out-of-vocabulary slang.
- **Sequence Classification**: Uses the `[CLS]` token representation passed through a classification head.
- **Optimization**: AdamW optimizer, warmup schedule, and evaluation metrics.


## 1. Setup & Transformers Import

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Computing Device: {device}")


## 2. Dataset Loading & Preprocessing

In [ ]:
train_path = os.path.join("dataset", "train.csv")
test_path = os.path.join("dataset", "test_1.csv")

if not os.path.exists(train_path):
    train_path = "train.csv"
if not os.path.exists(test_path):
    test_path = "test_1.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

t_col_train = "tweet" if "tweet" in train_df.columns else "text"
t_col_test = "tweet" if "tweet" in test_df.columns else "text"

def clean_bert_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = text.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_texts = [clean_bert_text(t) for t in train_df[t_col_train]]
train_labels = train_df["sarcastic"].astype(int).tolist()

test_texts = [clean_bert_text(t) for t in test_df[t_col_test]]
test_labels = test_df["sarcastic"].astype(int).tolist()

print(f"Train samples: {len(train_texts)} | Test samples: {len(test_texts)}")


## 3. BERT Tokenizer & Dataset
Tokenize using WordPiece tokenization with `[CLS]` and `[SEP]` tokens and attention masks.

In [ ]:
MODEL_NAME = "bert-base-uncased"
print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = BERTDataset(train_texts, train_labels, tokenizer)
test_dataset = BERTDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


## 4. BERT Sequence Classification Model Setup

In [ ]:
# Check if local fine-tuned BERT checkpoint exists; otherwise load bert-base-uncased
local_bert_dir = os.path.join("models", "bert")
if os.path.exists(os.path.join(local_bert_dir, "model.safetensors")) or os.path.exists(os.path.join(local_bert_dir, "pytorch_model.bin")):
    print(f"Loading local fine-tuned BERT checkpoint from {local_bert_dir}...")
    bert_model = AutoModelForSequenceClassification.from_pretrained(local_bert_dir).to(device)
else:
    print(f"Loading pretrained {MODEL_NAME}...")
    bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

print(bert_model.config)


## 5. Fine-Tuning BERT with AdamW & Linear Warmup

In [ ]:
# Compute class weighting
n_0 = train_labels.count(0)
n_1 = train_labels.count(1)
class_weights = torch.tensor([len(train_labels)/(2.0*n_0), len(train_labels)/(2.0*n_1)], dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer & Scheduler
epochs = 3
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

print("Starting BERT fine-tuning loop (3 epochs)...")
bert_model.train()
for ep in range(epochs):
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {ep+1}/{epochs} — Average Loss: {total_loss/len(train_loader):.4f}")

print("[OK] Fine-tuning completed.")


## 6. Evaluation on Official Test Split

In [ ]:
bert_model.eval()
bert_preds, bert_probs, bert_targets = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        bert_probs.extend(probs)
        bert_preds.extend((probs >= 0.5).astype(int))
        bert_targets.extend(labels.numpy())

bert_targets = np.array(bert_targets)
bert_preds = np.array(bert_preds)
bert_probs = np.array(bert_probs)

acc = accuracy_score(bert_targets, bert_preds)
prec, rec, f1, _ = precision_recall_fscore_support(bert_targets, bert_preds, average="binary", zero_division=0)
auc = roc_auc_score(bert_targets, bert_probs)

print("="*45)
print("              BERT Test Metrics")
print("="*45)
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1 Score:   {f1:.4f}")
print(f"ROC-AUC:    {auc:.4f}")
print("="*45)
print("\nClassification Report:\n", classification_report(bert_targets, bert_preds, target_names=["Non-Sarcastic", "Sarcastic"]))

cm = confusion_matrix(bert_targets, bert_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Sarcastic", "Sarcastic"],
            yticklabels=["Non-Sarcastic", "Sarcastic"])
plt.title("BERT Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


## 7. Threshold Tuning for BERT

In [ ]:
print("Analyzing classification threshold impact on BERT performance:")
for th in [0.4, 0.5, 0.6, 0.65, 0.7, 0.75]:
    p = (bert_probs >= th).astype(int)
    a = accuracy_score(bert_targets, p)
    pr, rc, f, _ = precision_recall_fscore_support(bert_targets, p, average="binary", zero_division=0)
    print(f"Threshold {th:.2f} -> Acc: {a:.4f}, Prec: {pr:.4f}, Rec: {rc:.4f}, F1: {f:.4f}")
